In [0]:
-- Q4: Additional Business Question - What is the customer loyalty profile?
-- Analyze customer behavior patterns to segment users by engagement and value

-- Customer Segmentation Analysis
WITH customer_metrics AS (
  SELECT
    o.user_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(op.product_id) AS total_products,
    ROUND(COUNT(op.product_id) * 1.0 / COUNT(DISTINCT o.order_id), 2) AS avg_basket_size,
    ROUND(SUM(op.reordered) * 100.0 / COUNT(*), 2) AS reorder_rate_pct,
    ROUND(AVG(o.days_since_prior_order), 1) AS avg_days_between_orders,
    MAX(o.order_number) AS customer_tenure_orders
  FROM instacart_gold.orders o
  INNER JOIN instacart_gold.order_products op ON o.order_id = op.order_id
  WHERE o.days_since_prior_order IS NOT NULL
  GROUP BY o.user_id
),
customer_segments AS (
  SELECT
    user_id,
    total_orders,
    total_products,
    avg_basket_size,
    reorder_rate_pct,
    avg_days_between_orders,
    customer_tenure_orders,
    CASE
      WHEN total_orders >= 50 AND reorder_rate_pct >= 60 THEN 'VIP Loyalist'
      WHEN total_orders >= 30 AND reorder_rate_pct >= 50 THEN 'Loyal Regular'
      WHEN total_orders >= 15 THEN 'Regular Customer'
      WHEN total_orders >= 5 THEN 'Occasional Shopper'
      ELSE 'New Customer'
    END AS customer_segment
  FROM customer_metrics
)
SELECT
  customer_segment,
  COUNT(*) AS num_customers,
  ROUND(AVG(total_orders), 1) AS avg_orders,
  ROUND(AVG(total_products), 1) AS avg_total_products,
  ROUND(AVG(avg_basket_size), 2) AS avg_basket_size,
  ROUND(AVG(reorder_rate_pct), 2) AS avg_reorder_rate_pct,
  ROUND(AVG(avg_days_between_orders), 1) AS avg_days_between_orders,
  ROUND(SUM(total_products) * 100.0 / (SELECT SUM(total_products) FROM customer_segments), 2) AS pct_total_revenue
FROM customer_segments
GROUP BY customer_segment
ORDER BY avg_orders DESC;